In [1]:
from lib import *
from sklearn.model_selection import StratifiedGroupKFold
import numpy as np
import pandas as pd

## for S00 only 

In [2]:
_, S00_exams_indices, N_S00_exams  = load_s00_conn_memmap()

In [3]:
meta = read_metadata()
labels = np.asarray(meta['pathology']).astype(int)[:N_S00_exams]
patients = np.asarray(meta['patient_id'])[:N_S00_exams]
sex = np.asarray(meta['female']).astype(int)[:N_S00_exams]
age = np.asarray(meta['age_group'])[:N_S00_exams]

Initiate the splitter

In [4]:
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=0)
folds_train_indieces, folds_val_indieces, folds_test_indieces = [], [], []

Perform the train-val-test split

In [5]:
for i, (train_index, test_index) in enumerate(sgkf.split(X=S00_exams_indices, y=labels, groups=patients)):
    y_tr, g_tr = labels[train_index], patients[train_index]
    inner = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=i)
    training, validation = next(iter(inner.split(np.zeros(len(train_index)), y_tr, g_tr)))

    folds_train_indieces.append(train_index[training])
    folds_val_indieces.append(train_index[validation])
    folds_test_indieces.append(test_index)

Check the results

In [6]:
print([(len(a), len(b), len(c)) for a, b, c in zip(folds_train_indieces, folds_val_indieces, folds_test_indieces)])

[(1085, 269, 343), (1090, 270, 337), (1093, 276, 328), (1078, 270, 349), (1091, 266, 340)]


Lekeage check

In [9]:
all_test = np.concatenate(folds_test_indieces)

for k, (tr, va, te) in enumerate(zip(folds_train_indieces, folds_val_indieces, folds_test_indieces)):
    # 1. no exam in two sets
    assert not (set(tr) & set(va)) and not (set(tr) & set(te)) and not (set(va) & set(te)), f"fold {k}: exam overlap"
    # 2. no patient in two sets (the important one)
    p_tr, p_va, p_te = set(patients[tr]), set(patients[va]), set(patients[te])
    assert not (p_tr & p_va), f"fold {k}: patient in train and val"
    assert not (p_tr & p_te), f"fold {k}: patient in train and test"
    assert not (p_va & p_te), f"fold {k}: patient in val and test"
    # 3. the three sets together cover every exam exactly once
    assert len(tr) + len(va) + len(te) == len(S00_exams_indices), f"fold {k}: sizes don't add up"
    # 4. class balance
    print(k, "train", np.bincount(labels[tr]), "val", np.bincount(labels[va]), "test", np.bincount(labels[te]))

# 5. across folds: every exam is tested exactly once
assert len(all_test) == len(set(all_test)) == len(S00_exams_indices), "test folds overlap or miss exams"
print("no leakage in splits")

0 train [567 518] val [147 122] test [163 180]
1 train [542 548] val [148 122] test [187 150]
2 train [569 524] val [129 147] test [179 149]
3 train [550 528] val [146 124] test [181 168]
4 train [563 528] val [147 119] test [167 173]
no leakage in splits


Save to file

In [10]:
np.savez("splits/s00.npz",
         **{f"fold{i}_{name}": arr
            for i, (a, b, c) in enumerate(zip(folds_train_indieces, folds_val_indieces, folds_test_indieces))
            for name, arr in (("train", a), ("val", b), ("test", c))})

Read from file per fold

In [15]:
s = np.load("splits/s00.npz")
k = 1
train_idx, val_idx, test_idx = s[f"fold{k}_train"], s[f"fold{k}_val"], s[f"fold{k}_test"]

In [16]:
train_idx

array([   1,    3,    4, ..., 1692, 1694, 1695], shape=(1090,))